In [ ]:
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE DATABASE FEATURE_ENGINEERING_DB;
CREATE OR REPLACE SCHEMA FEATURE_ENGINEERING_DB.RAW;
CREATE OR REPLACE SCHEMA FEATURE_ENGINEERING_DB.FEATURE_STORE;

USE DATABASE FEATURE_ENGINEERING_DB;
USE SCHEMA RAW;
CREATE OR REPLACE VIEW V_ORDERS AS
SELECT
    O_ORDERKEY       AS ORDER_ID,
    O_CUSTKEY        AS CUSTOMER_ID,
    O_TOTALPRICE     AS ORDER_AMOUNT,
    O_ORDERDATE      AS ORDER_DATE,
    O_ORDERSTATUS    AS ORDER_STATUS
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;
CREATE OR REPLACE TABLE CLEAN_ORDERS AS
SELECT
    CUSTOMER_ID,
    ORDER_ID,
    ORDER_AMOUNT,
    TO_DATE(ORDER_DATE) AS ORDER_DATE,
    ORDER_STATUS
FROM V_ORDERS
WHERE ORDER_STATUS IS NOT NULL
  AND ORDER_AMOUNT IS NOT NULL;
CREATE OR REPLACE TABLE RAW_ORDERS_1Y AS
SELECT *
FROM CLEAN_ORDERS
WHERE ORDER_DATE >= DATEADD(year, -30, CURRENT_DATE());
SET REF_DATE = CURRENT_DATE();

CREATE OR REPLACE TABLE CUSTOMER_FEATURES_BASE AS
SELECT
    CUSTOMER_ID,
    COUNT_IF(ORDER_DATE >= DATEADD(day, -30, $REF_DATE)) AS TOTAL_ORDERS_30D,
    COUNT_IF(ORDER_DATE >= DATEADD(day, -365, $REF_DATE)) AS TOTAL_ORDERS_365D,
    AVG(CASE WHEN ORDER_DATE >= DATEADD(day, -365, $REF_DATE)
             THEN ORDER_AMOUNT END) AS AVG_ORDER_AMOUNT_365D,
    MAX(ORDER_AMOUNT) AS MAX_ORDER_AMOUNT,
    DATEDIFF('day', MAX(ORDER_DATE), $REF_DATE) AS LAST_ORDER_DAYS_AGO
FROM RAW_ORDERS_1Y
GROUP BY CUSTOMER_ID;
CREATE OR REPLACE TABLE CUSTOMER_FEATURES AS
SELECT
    CUSTOMER_ID,
    TOTAL_ORDERS_30D,
    TOTAL_ORDERS_365D,
    AVG_ORDER_AMOUNT_365D,
    MAX_ORDER_AMOUNT,
    LAST_ORDER_DAYS_AGO,
    (AVG_ORDER_AMOUNT_365D - AVG(AVG_ORDER_AMOUNT_365D) OVER ())
        / NULLIF(STDDEV(AVG_ORDER_AMOUNT_365D) OVER (), 0)
        AS AVG_ORDER_AMT_365D_ZSCORE
FROM CUSTOMER_FEATURES_BASE;
USE SCHEMA FEATURE_ENGINEERING_DB.FEATURE_STORE;

CREATE OR REPLACE TABLE CUSTOMER_FEATURES_FS AS
SELECT
    CUSTOMER_ID AS ENTITY_CUSTOMER_ID,
    CURRENT_DATE() AS FEATURE_SNAPSHOT_DATE,
    TOTAL_ORDERS_30D,
    TOTAL_ORDERS_365D,
    AVG_ORDER_AMOUNT_365D,
    MAX_ORDER_AMOUNT,
    LAST_ORDER_DAYS_AGO,
    AVG_ORDER_AMT_365D_ZSCORE
FROM FEATURE_ENGINEERING_DB.RAW.CUSTOMER_FEATURES;
SELECT * FROM CUSTOMER_FEATURES_FS LIMIT 5;



In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer

# 1. Get session and load table
session = get_active_session()

df = session.table("FEATURE_ENGINEERING_DB.FEATURE_STORE.CUSTOMER_FEATURES_FS")
pdf = df.to_pandas()

print("Original shape:", pdf.shape)

# 2. Make sure key columns exist
required_cols = [
    "TOTAL_ORDERS_30D",
    "TOTAL_ORDERS_365D",
    "AVG_ORDER_AMOUNT_365D",
    "MAX_ORDER_AMOUNT",
    "LAST_ORDER_DAYS_AGO",
    "AVG_ORDER_AMT_365D_ZSCORE",
]

missing = [c for c in required_cols if c not in pdf.columns]
if missing:
    print("Missing required columns:", missing)
    raise SystemExit("Fix feature table before training.")

# 3. Drop rows where LAST_ORDER_DAYS_AGO is NaN (for label creation)
pdf = pdf.dropna(subset=["LAST_ORDER_DAYS_AGO"])
print("After dropping NaN LAST_ORDER_DAYS_AGO:", pdf.shape)

# 4. Create CHURN_FLAG using median of LAST_ORDER_DAYS_AGO
median_days = pdf["LAST_ORDER_DAYS_AGO"].median()
pdf["CHURN_FLAG"] = (pdf["LAST_ORDER_DAYS_AGO"] > median_days).astype(int)

print("Label distribution (based on LAST_ORDER_DAYS_AGO):")
print(pdf["CHURN_FLAG"].value_counts())

# 5. If still only one class, fallback to MAX_ORDER_AMOUNT
if pdf["CHURN_FLAG"].nunique() < 2:
    print("\nOnly one class after first attempt, using fallback label based on MAX_ORDER_AMOUNT...")
    pdf = pdf.dropna(subset=["MAX_ORDER_AMOUNT"])
    median_amt = pdf["MAX_ORDER_AMOUNT"].median()
    pdf["CHURN_FLAG"] = (pdf["MAX_ORDER_AMOUNT"] > median_amt).astype(int)
    print("Fallback label distribution:")
    print(pdf["CHURN_FLAG"].value_counts())

# 6. Final check: do we have 2 classes now?
if pdf["CHURN_FLAG"].nunique() < 2:
    print("\n❌ Still only one class in CHURN_FLAG. Cannot train a classifier on a single class.")
    print("Try changing the label logic or using a different feature to define churn.")
else:
    print("\n✅ Final label distribution:")
    print(pdf["CHURN_FLAG"].value_counts())

    # 7. Prepare X and y
    X = pdf[required_cols]
    y = pdf["CHURN_FLAG"]

    # 8. Impute missing values in features
    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    # 9. Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, random_state=42, stratify=y
    )

    # 10. Train Logistic Regression
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # 11. Predictions and report
    preds = model.predict(X_test)
    print("\n===== MODEL TRAINED SUCCESSFULLY =====")
    print(classification_report(y_test, preds))


In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# -------------------------------
# 1️⃣ CONFUSION MATRIX
# -------------------------------
cm = confusion_matrix(y_test, preds)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", ax=ax)
plt.title("Confusion Matrix")
plt.show()

# -------------------------------
# 2️⃣ ROC CURVE + AUC
# -------------------------------
y_prob = model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()


# -------------------------------
# 3️⃣ PRINT METRICS
# -------------------------------
acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds)
rec = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)

print("===== METRICS SUMMARY =====")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {roc_auc:.4f}")
